# Regression

This notebook is a hands-on introduction to regression: how to estimate regression models, interpret their output, and generate the supporting visualizations. We start with **correlation**, build up to **simple and multiple regression**, put the tools to work estimating stock **betas**, and finish with **logistic regression** for binary outcomes. 

> **A note on scope.** This lesson is a simple, practical introduction to correlation and regression. Regression can fill an entire course (or several), and this lesson is a starting point, not a replacement for that kind of in-depth study.

Before starting, work through the [Data Visualization](../05_data_viz/data_visualization.ipynb) notebook. This module assumes you are comfortable loading, summarizing, and plotting a dataset with `pandas` and `seaborn`.

## General housekeeping items

As in the previous notebooks, select your kernel before running anything. Click **Select Kernel** in the top right of this notebook, choose **Python Environments...**, and select the `.venv` environment you created in [Getting Started](../01_getting_started/course_intro.ipynb).

Let's begin by importing the necessary libraries. `pandas`, `numpy`, `seaborn`, and `matplotlib` are the same packages you have used in earlier modules. The new arrival is `statsmodels`, the statistics library we use for regression in this course: `smf` gives us the model-fitting functions (`ols()` for linear regression and `logit()` for logistic regression), and `summary_col` collects several regression results into one comparison table. We also import `display` and `HTML` to render a results table as HTML. Finally, we apply the same seaborn theme we used in [Data Visualization](../05_data_viz/data_visualization.ipynb), so the plots here look like the ones you have already seen.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from IPython.display import display, HTML

sns.set_theme(style="whitegrid")

## Correlation

A correlation measures the **association** between two variables, or how they move together. Positive values indicate a positive association (when one variable is high, the other tends to be high), negative values indicate a negative association, and the measure always falls between -1 and 1. A value near zero means the two variables tell us little about each other, at least linearly. 

This is not our first brush with the idea: in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) we plotted diamond price against carat weight and watched the two rise together. Correlation turns that visual impression into a single number.

**Free references:**
- [Correlation – Wikipedia](https://en.wikipedia.org/wiki/Correlation)
- [How to Calculate a Pearson Correlation Coefficient by Hand](https://www.statology.org/correlation-coefficient-by-hand/)

### A running example: Alabama football

To see how correlation is calculated, let's compute one correlation entirely by hand, the same way we computed means and variance with the college football coaches in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb). Consider the following data:

| Date | Opponent | Result | Total yards ($x$) | Points ($y$) |
|----------|--------------------|---------|-------|----|
| 09/05/26 | East Carolina      | W 48-10 | 487 | 48 |
| 09/12/26 | @ Kentucky         | W 45-17 | 343 | 45 |
| 09/19/26 | Florida State      | W 50-36 | 567 | 50 |
| 09/26/26 | South Carolina     | W 49-18 | 489 | 49 |
| 10/03/26 | @ Mississippi St.  | W 56-23 | 483 | 56 |
| 10/10/26 | Georgia            | L 17-20 | 346 | 17 |
| 10/17/26 | @ Tennessee        | L 10-26 | 201 | 10 |

Source: [cfbstats.com](http://www.cfbstats.com/), Alabama total offense game log, 2026 season as of 10/17/2026.

The dataset is small enough to type in directly, so let's build it in the notebook rather than loading a file. Throughout this example, $N = 7$ refers to the number of games:

In [ ]:
bama_games = pd.DataFrame({
    'opponent': ['East Carolina', 'Kentucky', 'Florida State',
                 'South Carolina', 'Mississippi St.', 'Georgia',
                 'Tennessee'],
    'total_yards': [487, 343, 567, 489, 483, 346, 201],
    'points': [48, 45, 50, 49, 56, 17, 10]
})

bama_games

#### Step 1: the means

Every formula in this example is built from deviations around the means, so we compute those first:

$$
\bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i = \frac{2{,}916}{7} \approx 416.57
\qquad
\bar{y} = \frac{1}{N}\sum_{i=1}^{N} y_i = \frac{275}{7} \approx 39.29
$$

These are the same by-hand mean calculations we did for the coaches' titles in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb); the bar notation ($\bar{x}$, read "x-bar") is just shorthand for the mean of $x$.

#### Step 2: the deviation table

Next, for each game we compute how far $x$ and $y$ sit from their means, the product of those deviations, and the squared deviations. The products are the heart of correlation: a game where yards and points are both above average (or both below average) contributes a positive product, and a game where they move in opposite directions contributes a negative one. The two squared-deviation columns should look familiar as well: they are the same squared deviations that drive the **variance** and **standard deviation** from [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb).

| Opponent | $x$ | $y$ | $x-\bar{x}$ | $y-\bar{y}$ | $(x-\bar{x})(y-\bar{y})$ | $(x-\bar{x})^2$ | $(y-\bar{y})^2$ |
|-----------------|-----|----|---------|--------|-----------|------------|----------|
| East Carolina   | 487 | 48 | 70.43   | 8.71   | 613.73    | 4,960.18   | 75.94    |
| Kentucky        | 343 | 45 | -73.57  | 5.71   | -420.41   | 5,412.76   | 32.65    |
| Florida State   | 567 | 50 | 150.43  | 10.71  | 1,611.73  | 22,628.76  | 114.80   |
| South Carolina  | 489 | 49 | 72.43   | 9.71   | 703.59    | 5,245.90   | 94.37    |
| Mississippi St. | 483 | 56 | 66.43   | 16.71  | 1,110.31  | 4,412.76   | 279.37   |
| Georgia         | 346 | 17 | -70.57  | -22.29 | 1,572.73  | 4,980.33   | 496.65   |
| Tennessee       | 201 | 10 | -215.57 | -29.29 | 6,313.16  | 46,471.04  | 857.65   |
| **Sum**         | 2,916 | 275 | 0   | 0      | **11,504.86** | **94,111.71** | **1,951.43** |

Entries are rounded to two decimals; the sums are computed from the unrounded values. Notice that the deviations themselves always sum to zero; that is what it means to deviate around the mean. Notice also that six of the seven products are positive: high-yard games were mostly high-scoring games.

#### Step 3: correlation by formula

The correlation coefficient scales the sum of the products by the spread of each variable, which is what forces $r$ to land between -1 and 1:

$$
r = \frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2 \sum(y_i-\bar{y})^2}}
  = \frac{11{,}504.86}{\sqrt{94{,}111.71 \times 1{,}951.43}}
  \approx 0.849
$$

A correlation of about 0.85 is a strong positive association: games with more offensive yards tend strongly to be games with more points. Notice what the formula is doing: the numerator asks whether yards and points move together, while the denominator scales the answer by each variable's own spread, the spread the standard deviation measures. Correlation is co-movement, measured relative to how much each variable varies on its own. Let's confirm the hand calculation with `corr()`. Note that the default correlation method in `corr()` is **Pearson's r**, which is the focus of this exercise.

In [ ]:
bama_games[['total_yards', 'points']].corr()

Hand math and `corr()` agree: $r \approx 0.849$. Keep the deviation table handy, because the same sums reappear when we fit a regression to this data below.

## Correlation on simulated data

A single correlation is just one number. To build intuition for what different *strengths* of correlation look like, let's import a set of simulated datasets containing two correlated variables, `x` and `y`. Each dataset has 100 observations. The first three datasets illustrate a positive correlation based on the equation $y = x + \text{noise}$, where the amount of 'random' noise varies across the datasets: little, some, and more noise. The final dataset illustrates a negative correlation based on the equation $y = 100 - x + \text{noise}$. All four datasets are 'stacked' in one file and can be identified by the `label` column (e.g., `Sim 1: Positive - Little noise`).

In [ ]:
sim_data = pd.read_csv('sim_data.csv')

### Plot the data and determine correlations

Correlation is sometimes easiest to understand visually, so let's start by plotting `y` on `x` separately for each simulated dataset. Note that we use `relplot` with `col` to create a faceted plot, one panel per dataset, as we did in [Data Visualization](../05_data_viz/data_visualization.ipynb):

In [ ]:
g = sns.relplot(data=sim_data,
                x="x",
                y="y",
                kind="scatter",
                col="label",
                col_wrap=2,
                height=3,
                aspect=1.2
                )

for ax, label in zip(g.axes.flat, g.col_names):
    group = sim_data[sim_data['label'] == label]
    r = group['x'].corr(group['y'])
    ax.text(0.97, 0.03, f'r = {r:.3f}',
            transform=ax.transAxes,
            ha='right', va='bottom',
            fontsize=9)

g.figure.suptitle('Simulated Datasets: y versus x', y=1.03)

plt.show()

The first three datasets above exhibit positive correlations, with the points scattering more widely as the noise grows, and the fourth dataset exhibits a negative correlation. Notice how the correlation weakens as the noise grows, even though the underlying relation between `x` and `y` is the same.

## Regression: fitting a trend line to the data

### Linear regression overview

Regression is perhaps the most powerful and flexible tool in a data analyst's toolkit. It is simple to estimate with modern statistical packages, it is fast even on large datasets, and it scales from one explanatory variable to many.

We will primarily use the **ordinary least squares** (OLS) method to estimate regression models. OLS finds the intercept and coefficient(s) that minimize the **sum of the squared errors**, which is what it means to "fit" a model to the data.

You have seen the underlying idea before. In 8th grade math, it is the slope-intercept equation:

$$y = mx + b$$

where $y$ is an outcome, $m$ is the slope, $x$ is the explanatory variable, and $b$ is the intercept (or constant). This equation is so well known that it doubles as [crowd control advice](https://www.causeweb.org/cause/sites/default/files/resources/fun/cartoons/MathConcert2013.jpg). In grad school math, we will use a similar regression equation complete with Greek letters:

$$y = \beta_0 + \beta_1 x_1 + \varepsilon$$

where $\beta_0$ (beta naught) is the intercept, $\beta_1$ (beta one) is the coefficient (slope), and $\varepsilon$ is the **unexplained variation**, also called the error, noise, or residual. The other two variables go by several names as well: the outcome $y$ is also called the dependent, explained, or response variable (or the regressand), and $x$ is also called the independent, explanatory, or predictor variable (or the regressor).

Here is what the equation looks like for a concrete example, using the diamonds data:

$$price = \beta_0 + \beta_1\, carat + \varepsilon$$

In Python, we estimate this model with the `ols()` function from `statsmodels`: `ols("price ~ carat", data=diamonds)`. We will walk through the output of this regression shortly.

Two features make regression especially flexible:

- **Many variables.** A model can include several predictors at once, which we will do in the multiple regression section below:

$$price = \beta_0 + \beta_1\, carat + \beta_2\, clarity + \varepsilon$$

- **Non-linear trends.** A *linear* regression does not require *linear* variables per se. The model can include transformations of variables, such as squared terms or natural logs:

$$price = \beta_0 + \beta_1\, carat + \beta_2\, carat^2 + \varepsilon$$

Finally, pay attention to what kind of variable each regressor is, because the type determines how it should enter the model:

- **Numeric variables** contain numbers with numeric meaning (1 < 2, and 2 − 1 = 4 − 3). Examples include population, company assets, and earnings per share.
- **Indicator variables** take binary (0/1) values. Examples include bankruptcy status and employment status.
- **Categorical variables** represent categories, such as country or industry. Many categorical variables are *coded* using numbers (e.g., zip codes and industry classifications), but they should still be treated as categorical. When the categories are not ordinal, each category should enter the model as a separate indicator variable rather than as one numeric variable (more on this later in the notebook).

### A simple regression by hand: Alabama football

Before we let the computer do the work, let's fit one regression ourselves using the Alabama deviation table from the correlation section. For a single explanatory variable, the OLS slope has a closed-form solution, and it shares its numerator with the correlation: the sum of the deviation products. Only the denominator changes:

$$
b_1 = \frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sum(x_i-\bar{x})^2}
    = \frac{11{,}504.86}{94{,}111.71}
    \approx 0.122
$$

The slope has units: about 0.122 points per yard, or roughly **12 more points per additional 100 yards** of offense. The intercept then anchors the line so that it passes through the point $(\bar{x}, \bar{y})$:

$$b_0 = \bar{y} - b_1 \bar{x} = 39.29 - 0.122 \times 416.57 \approx -11.64$$

As in the deviation table, this arithmetic carries the unrounded slope (0.122247), so redoing it with the rounded 0.122 gives -11.53 instead. The same applies to the estimates below, which differ in the first decimal place from what the rounded coefficients produce; the computer output uses full precision throughout.

Do not interpret the intercept literally: it is the predicted score of a game with zero offensive yards, which sits far outside the range of our data (more on the dangers of extrapolating shortly). Putting the pieces together, our fitted equation is:

$$\hat{y} = -11.64 + 0.122\, x$$

Two connections back to correlation are worth seeing. First, for a regression with one explanatory variable, the r-squared (the share of the variation in $y$ that the model explains) is literally the correlation squared:

$$R^2 = r^2 = 0.849^2 \approx 0.72$$

That is, about 72% of the variation in points is explained by total yards. Second, the slope is the correlation rescaled by the two sample standard deviations from [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) ($s_y = 18.03$ points and $s_x = 125.24$ yards), which shows that correlation and the regression slope carry the same information in different units:

$$b_1 = r \cdot \frac{s_y}{s_x} = 0.849 \times \frac{18.03}{125.24} \approx 0.122$$

If our hand calculations are right, `ols()` must reproduce them exactly. Let's estimate a formal regression and evaluate the output below.

In [ ]:
bama_reg = smf.ols("points ~ total_yards", data=bama_games).fit()

print(bama_reg.summary())

This agrees with our arithmetic (but is much simpler!). Finally, let's draw the scatter plot using `lmplot`, which fits and draws the same OLS line we just computed by hand, and label each point by opponent. We will come back to a couple of these games when we discuss residuals:

In [ ]:
sns.lmplot(data=bama_games,
           x='total_yards',
           y='points',
           ci=None,
           height=6,
           aspect=1.2
           )

for _, row in bama_games.iterrows():
    plt.annotate(row['opponent'],
                 (row['total_yards'], row['points']),
                 xytext=(6, 6),
                 textcoords='offset points',
                 fontsize=9)

plt.title('Alabama 2026: Points Scored vs. Total Offensive Yards')
plt.xlabel('Total Offensive Yards')
plt.ylabel('Points Scored')

plt.show()

### Add a trendline with regression

We will not compute slopes by hand again; from here, the computer does the fitting. Let's try this with the [diamonds](https://ggplot2.tidyverse.org/reference/diamonds.html) dataset, which we load from the pickle file in this folder as in earlier modules. As in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb), we keep diamonds under 1.5 carats, stored here in `diamonds_filtered`, and draw the regression line over the scatter with `lmplot`:

In [ ]:
diamonds = pd.read_pickle('diamonds.pkl')

diamonds_filtered = diamonds[diamonds['carat'] < 1.5]

sns.lmplot(data=diamonds_filtered,
           x='carat',
           y='price',
           ci=None,
           scatter_kws={'alpha': 0.2},
           line_kws={'color': 'red'},
           height=6,
           aspect=1.2
           )

plt.title('Scatter Plot of Price versus Carat')
plt.xlabel('Carat')
plt.ylabel('Price')

plt.show()

#### Visualization practice

Before moving on, make sure you can connect the chart to the equation. Can you relate each element of the graph above to the following model?

$$price = \beta_0 + \beta_1\, carat + \varepsilon$$

Where on the chart would you find $\beta_0$? $\beta_1$? $\varepsilon$? $price$? $carat$?

## Regression output with the diamonds data

In the Alabama example, we printed the output of `ols()`. Learning to read it is a skill in itself. Recall that we imported the `statsmodels` formula interface as `smf` in the housekeeping section; the library contains several useful features for estimating regressions, collecting output, and presenting results. Below, we scale up from seven games to roughly 47,000 diamonds and examine the complete regression summary, regressing price (the outcome) on carat weight (the explanatory variable):

In [ ]:
diamonds_reg_1 = smf.ols("price ~ carat", data=diamonds_filtered).fit()

print(diamonds_reg_1.summary())

## Interpreting the regression output

The summary above packs in a lot, so here is a quick map of the pieces that matter most:

- **Coefficients (`coef`):** the estimated intercept and slope(s), the fitted values of $\beta_0$ and $\beta_1$
- **Standard errors (`std err`):** the precision of each coefficient estimate. Despite the name, a standard error is *not* the same thing as the error term $\varepsilon$ in the regression equation.
- **Significance indicators (`t`, `P>|t|`, and the stars in the compact tables below):** tools of statistical inference, which we take up in the next module.
- **R-squared:** the model's explanatory power, formally called the "coefficient of determination."

For now, focus on the coefficient column: it gives the estimated intercept and slope, which we can read directly into our regression equation:

$$price = -1{,}758 + 6{,}859\, carat + \varepsilon$$

Plugging values into the fitted equation produces predictions. The predicted price of a one-carat diamond:

$$price = -1{,}758 + 6{,}859\,(1) = 5{,}101$$

The predicted price of a 0.2-carat diamond:

$$price = -1{,}758 + 6{,}859\,(0.2) = -386$$


### Discussion questions
A diamond with a negative price? Why might the model predict that? How do you interpret the intercept?

What do we predict happens to price if size increases by one carat?

### Other ways to output regression statistics
Notice the simple syntax for estimating a regression once the data is tidy and properly formatted. Notice also that we stored the regression results in `diamonds_reg_1`, which we can reference later. For example, we can present the results in a compact table using `summary_col`:

In [ ]:
results_table = summary_col(diamonds_reg_1,
                            stars=True,
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

We can also render the same table as HTML, which is handy when you want to copy results into a report or a slide deck:

In [ ]:
html_table = results_table.as_html()

display(HTML(html_table))

## What is regression good for?

Regressions can be used to **predict** things. Suppose we want to predict NFL playoff appearances, and we have historical data on football teams including playoff status, total budget, player salaries, coaching, location, and so forth. A regression turns that history into a prediction. What else would we like to predict?

Regressions can also be used to **test relations** between variables. Suppose we want to know whether hiring a more expensive football coach will increase the likelihood of making the playoffs. What other relations might we want to test?

## Regression points of caution

Regression is powerful, but it has some important weaknesses:

- **Sensitive to outliers**... just like the means we discussed in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb).
- **Data and model dependent.** Regression estimates are only as good as the model and the data... garbage in → garbage out.
- **Interpret estimates within the range of the data.** That is, use care when extrapolating beyond the values you actually observed. For a humorous illustration, see the XKCD comic [*Extrapolating*](https://imgs.xkcd.com/comics/extrapolating.png).

## Regression illustration with Anscombe's quartet

Let's import a series of datasets called **Anscombe's quartet**. We met this data in [Getting Started](../01_getting_started/course_intro.ipynb); it is a classic example used to illustrate the importance of visualizing data before (or at least alongside) statistical analysis.

In [ ]:
ansc_quar = pd.read_pickle('anscombe.pkl')

This file is made up of four example datasets, each identified by the `dataset` column. Let's plot each one with its fitted regression line:

In [ ]:
grid = sns.lmplot(data=ansc_quar,
                  x="x",
                  y="y",
                  ci=None,
                  line_kws={'color': 'red'},
                  truncate=False,
                  col="dataset",
                  col_wrap=2,
                  height=3,
                  aspect=1.2
                  )

grid.figure.suptitle("Anscombe's Quartet with OLS Regression Lines", y=1.03)

plt.show()

The four scatter plots above look nothing alike. Now let's look at the regression output in table form for easy comparison. Here, we estimate a separate regression for each dataset and then aggregate the output in one table:

In [ ]:
anscombes_reg_1 = smf.ols("y ~ x",
                          data=ansc_quar[ansc_quar['dataset'] == 'I']).fit()
anscombes_reg_2 = smf.ols("y ~ x",
                          data=ansc_quar[ansc_quar['dataset'] == 'II']).fit()
anscombes_reg_3 = smf.ols("y ~ x",
                          data=ansc_quar[ansc_quar['dataset'] == 'III']).fit()
anscombes_reg_4 = smf.ols("y ~ x",
                          data=ansc_quar[ansc_quar['dataset'] == 'IV']).fit()

results_table = summary_col([anscombes_reg_1, anscombes_reg_2,
                             anscombes_reg_3, anscombes_reg_4],
                            stars=True,
                            model_names=['I', 'II', 'III', 'IV'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)


## Regression with non-linear variables

Look closely at the table above: four very different datasets produce essentially identical intercepts, slopes, and r-squared values. Regressions are great at fitting lines to our data, and while the estimates are 'linear', the variables need not be. That is, a regression model is flexible to non-linear transformations of variables (e.g., squared or logged). In Anscombe's quartet, dataset II follows a non-linear trend between `y` and `x`. Let's visualize a quadratic model that includes a squared term:

In [ ]:
sns.lmplot(data=ansc_quar[ansc_quar['dataset'] == 'II'],
           x="x",
           y="y",
           order=2,
           ci=None,
           line_kws={'color': 'red'},
           height=6,
           aspect=1.2
           )

plt.title('Anscombe\'s Quartet - Dataset II - Quadratic')

plt.show()

Now let's estimate the regression using the `I(x**2)` term. Recall from [Python Basics](../02_python_basics/python_basics.ipynb) that `**` is used for exponentiation. `I()` allows us to perform a basic calculation on a variable and include it as a regressor, which is a useful technique for modeling non-linear relationships. Notice in the output how well the quadratic model fits dataset II:

In [ ]:
anscombes_reg_5 = smf.ols("y ~ x + I(x**2)",
                          data=ansc_quar[ansc_quar['dataset'] == 'II']).fit()

print(anscombes_reg_5.summary())

Anscombe's quartet illustrates why we need to be careful in our analyses. Very different patterns can produce the same regression estimates, so make sure to:

- Plot and visualize your data.
- Consider non-linear or unmodeled relationships.
- Investigate the effect of outliers.

## Unexplained variation: residuals, noise, or error

Recall that, in most settings, we do not know or measure all of the factors that predict outcomes. In other words, our regression model is unlikely to perfectly predict the outcome, and the gap between each observed value and the fitted line is the **residual**, the $\varepsilon$ in our regression equation. A useful way to think about the residual: it bundles together everything that influences the outcome but is not captured by the variables in the model (omitted factors, measurement error, and plain randomness). That also means the residual is defined relative to a particular model: add a variable that matters, and part of what used to sit in the residual moves into the fitted equation. A fitted `statsmodels` model hands you these pieces directly: `.fittedvalues` holds the model's predictions for each observation in the estimation sample, and `.resid` holds each observation's residual.

### Visualizing residuals with Anscombe's quartet

To see every residual in a dataset at once, let's return to Anscombe's quartet. Below, we draw datasets I and III with the fitted line in red and each residual as a vertical blue line:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, dataset_label in zip(axes, ['I', 'III']):
    data = ansc_quar[ansc_quar['dataset'] == dataset_label].copy()
    reg = smf.ols("y ~ x", data=data).fit()
    data['fitted'] = reg.fittedvalues

    for _, row in data.iterrows():
        ax.plot([row['x'], row['x']], [row['fitted'], row['y']],
                color='steelblue', linewidth=1.2, zorder=2)

    x_range = np.linspace(data['x'].min() - 0.5, data['x'].max() + 0.5, 100)
    ax.plot(x_range,
            reg.params['Intercept'] + reg.params['x'] * x_range,
            color='red', linewidth=1.5, zorder=3)

    ax.scatter(data['x'], data['y'],
               color='white', edgecolors='black', s=40, zorder=4)

    ax.set_title(f'Dataset {dataset_label}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    sns.despine(ax=ax)

plt.suptitle("Anscombe's Quartet: Residuals Highlighted", y=1.02)
plt.tight_layout()
plt.show()

In dataset I, the residuals look like random noise: some points sit above the line, some below, with no obvious pattern. Dataset III tells a richer story: ten of its eleven points fall almost exactly on a straight line, but the single outlier tilts the fitted line away from them, so even the well-behaved points pick up residuals. And notice that those residuals are not random: they drift steadily from positive to negative as $x$ increases. 

### A teaching moment: the Kentucky game

Look back at the Alabama scatter plot above. The Kentucky game produced 343 yards but 45 points, while the fitted equation predicts $\hat{y}(343) = -11.64 + 0.122 \times 343 \approx 30$ points, so Kentucky sits about **15 points above the line**. That gap is the game's residual or $\varepsilon$. Now compare it to the Georgia game: almost identical yardage (346), yet only 17 points, a residual of about 14 points *below* the line. 

Why? Because points do not come only from total offensive yards. Defensive scores, field position, special teams, and red-zone efficiency all move the score in ways total yards cannot capture. Every one of those influences is in the residual, the model's catch-all for whatever determines the score that total yards does not measure. Our model explains about 72% of the variation in points. The other 28% is the unexplained variation, and the big misses in the Kentucky and Georgia games account for most of it (at least in this case).

This is also a good moment to repeat the mantra: **correlation is not causation.** When a regression or correlation can and cannot support a causal claim is the subject of the [next module](../07_causal_statistical_inference). What other variables would you add to this model to shrink those residuals? We will see how to include them in the multiple regression section below.

## Natural log transformations in regression

A change in natural log represents an approximate percentage change. For example, an investment increasing from 100 dollars to 110 dollars is a 10% increase, and $\ln(110) - \ln(100) = 0.0953$, or roughly 10%. The approximation works well for relatively small percentage changes; the precise conversion is $e^{0.0953} - 1 = 0.10$, or exactly 10%.

This interpretation is part of what makes logs so useful. When both the y and x variables are in log form, the coefficient represents the effect of a percentage change in x on the percentage change in y (sometimes called an 'elasticity'). One caution: you cannot take the log of 0 or negative values. For zeros, there is no perfect way to deal with this, but it is common to add a small value (1 or smaller) to the variable before taking the log. Negative numbers need a different approach altogether.

Logs earn their keep in other ways too. When data grows exponentially, taking the log often turns a curved relation into a straight line that a linear regression can fit well, as we are about to see with the diamonds. Logging also compresses extreme values and reins in the long right tails. Also, it tends to stabilize the spread of the data: notice in the scatter plot above how the variation in price grows as carat grows. Statisticians call that pattern **heteroskedasticity**, and logging often tames it.

**Free references:**
- [The Logarithm Transformation](https://people.duke.edu/%7Ernau/411log.htm)
- [Multiple Regression Analysis](http://fmwww.bc.edu/ec-c/F2007/228/EC228.f2005.nn06.pdf)

Log transformations of variables are often useful in improving model fit and interpretability (among other benefits). Recall from the scatter plots above that the relation between price and carat in the diamonds data curves upward. Let's redo our regression using log values for price and carat. To start, we calculate the logs, store them in new variables, and plot:

In [ ]:
diamonds_filtered = diamonds_filtered.copy()
diamonds_filtered['log_carat'] = np.log(diamonds_filtered['carat'])
diamonds_filtered['log_price'] = np.log(diamonds_filtered['price'])

sns.lmplot(data=diamonds_filtered,
           x='log_carat',
           y='log_price',
           ci=None,
           scatter_kws={'alpha': 0.2},
           line_kws={'color': 'red'},
           height=6,
           aspect=1.2
           )

plt.title('Scatter Plot of Price versus Carat (Log Form)')
plt.xlabel('Natural Log of Carat')
plt.ylabel('Natural Log of Price')

plt.show()

The relation now looks roughly linear. Let's estimate the regression in log form. Rather than using the new columns, we can apply `np.log()` directly inside the formula:

In [ ]:
diamonds_reg_2 = smf.ols("np.log(price) ~ np.log(carat)",
                         data=diamonds_filtered).fit()

print(diamonds_reg_2.summary())

The log-log specification changes how we read the output. Using values from the regression:

$$\ln(price) = 8.458 + 1.692\, \ln(carat) + \varepsilon$$

The coefficient of 1.692 is an elasticity: a 1% increase in carat weight predicts roughly a 1.7% increase in price. Predictions now take one extra step, because the equation predicts the *log* of price; exponentiating the result converts it back to dollars and gives an approximate predicted price. For a one-carat diamond, the model approximates the price as:

$$\ln(price) = 8.458 + 1.692 \ln(1) = 8.458 \quad\Rightarrow\quad price = e^{8.458} \approx 4{,}712$$

For a 0.2-carat diamond:

$$\ln(price) = 8.458 + 1.692 \ln(0.2) = 8.458 - 2.723 = 5.735 \quad\Rightarrow\quad price = e^{5.735} \approx 309$$

Notice that the troublesome negative prediction from the earlier model is gone.

## Multiple regression

Regressions can have more than one predictor, or 'x', variable. In fact, we can expand our regression models to estimate the effects of many variables at once. Let's add all four of the diamond "C's" (carat, clarity, cut, and color) to the model. Since clarity, cut, and color are ordered categories, we first convert them to numeric codes (multiplied by -1 so that higher values are associated with better characteristics). Then we estimate a regression for each variable on its own, followed by one with all four together, and collect the five regressions in one comparison table, numbered (1) through (5):

In [ ]:
# Create numeric versions of the categorical columns: clarity, cut, and color.
for column in ['clarity', 'cut', 'color']:
    codes = pd.Categorical(diamonds_filtered[column]).codes
    diamonds_filtered[f'{column}_numeric'] = codes * -1

diamonds_reg_short1 = smf.ols("np.log(price) ~ np.log(carat)",
                              data=diamonds_filtered).fit()
diamonds_reg_short2 = smf.ols("np.log(price) ~ clarity_numeric",
                              data=diamonds_filtered).fit()
diamonds_reg_short3 = smf.ols("np.log(price) ~ cut_numeric",
                              data=diamonds_filtered).fit()
diamonds_reg_short4 = smf.ols("np.log(price) ~ color_numeric",
                              data=diamonds_filtered).fit()
# Python joins the two adjacent strings below into one formula.
diamonds_reg_long1 = smf.ols("np.log(price) ~ np.log(carat) + clarity_numeric"
                             " + cut_numeric + color_numeric",
                             data=diamonds_filtered).fit()

results_table = summary_col([diamonds_reg_short1, diamonds_reg_short2,
                             diamonds_reg_short3, diamonds_reg_short4,
                             diamonds_reg_long1],
                            stars=True,
                            model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)


Before reading the table, it helps to know what the numeric codes mean. Each runs from 0 for the best category downward, so a higher (less negative) code is a better diamond. You will need this mapping to plug a diamond of your own into the fitted equation below:


In [ ]:
for column in ['clarity', 'cut', 'color']:
    categories = diamonds_filtered[column].cat.categories
    mapping = {category: -code for code, category in enumerate(categories)}
    print(f"{column}: {mapping}")


Note the patterns in the coefficient estimates and r-squared values above. Why do clarity, cut, and color have *negative* coefficients when each enters the model alone (columns (2)-(4))? Why do these reverse when all variables are included simultaneously (column (5))?

Regression models can include many predictor variables, not just one y and one x, which allows us to develop predictions based on multiple dimensions. Using the values from column (5) of the numeric-codes table above:

$$\ln(price) = 9.244 + 1.886\, \ln(carat) + 0.121\, clarity + 0.031\, cut + 0.076\, color$$

## Multiple regression with categorical variables

Above, we treated all variables as numeric, even ordinal categorical variables like clarity. Alternatively, we can estimate the effect of each level of a categorical variable separately by including an indicator variable for each level. In `statsmodels`, wrapping a variable in `C()` does this automatically. See the example below:

In [ ]:
diamonds_reg_long2 = smf.ols(
    "np.log(price) ~ np.log(carat) + C(clarity) + C(cut) + C(color)",
    data=diamonds_filtered).fit()

results_table = summary_col(diamonds_reg_long2,
                            stars=True,
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

Now look at the `C()` output directly above. Notice that each categorical variable is split into several 'indicator' variables, one per category. If you pay close attention, one category of each variable is excluded (IF for clarity, Ideal for cut, and D for color): that category is the 'base' or comparison group, and its effect is absorbed into the intercept. In any case, the model's fit (the r-squared) was already very high (>0.97), and this more flexible approach results in only a modest improvement (0.978 vs. 0.975). Note that this measures how well the model fits the data it was estimated on; whether it would predict *new* diamonds this well is a different question, and one we take up in the [Machine Learning module](../08_machine_learning).

## Extra practice with regression: CPUs

In this folder, you will find a dataset containing prices and performance metrics of consumer CPUs. The data was collected from [PassMark](https://www.cpubenchmark.net/) in September 2022. Let's practice regression and visualization using this dataset:

In [ ]:
cpu_data = pd.read_csv('cpu_data.csv')

cpu_data.head(10)

First, let's plot CPU price on CPU performance with a regression line:

In [ ]:
sns.lmplot(data=cpu_data,
           x='cpu_mark',
           y='price_usd',
           scatter_kws={'alpha': 0.5},
           line_kws={'color': 'red'},
           ci=None,
           height=6,
           aspect=1.2
           )

plt.title("CPU Price vs. CPU Performance")
plt.xlabel("CPU Performance")
plt.ylabel("CPU Price")

plt.show()

Note that the relationship is non-linear, a lot like the diamonds data above. Let's try logging both variables to see if that improves linearity:

In [ ]:
cpu_data['log_price'] = np.log(cpu_data['price_usd'])
cpu_data['log_performance'] = np.log(cpu_data['cpu_mark'])

sns.lmplot(data=cpu_data,
           x='log_performance',
           y='log_price',
           ci=None,
           scatter_kws={'alpha': 0.5},
           line_kws={'color': 'red'},
           height=6,
           aspect=1.2
           )

plt.title("CPU Price vs. CPU Performance (Log-Log Scale)")
plt.xlabel("Log CPU Performance")
plt.ylabel("Log CPU Price")

plt.show()

That's better! Next, let's assign color to brand to see whether AMD and Intel have different pricing (accounting for performance).

In [ ]:
sns.lmplot(data=cpu_data,
           x='log_performance',
           y='log_price',
           hue='brand',
           ci=None,
           scatter_kws={'alpha': 0.5},
           height=6,
           aspect=1.2
           )

plt.title("CPU Price vs. CPU Performance (Log-Log Scale)")
plt.xlabel("Log CPU Performance")
plt.ylabel("Log CPU Price")

plt.show()

Now let's estimate the regressions, building from performance alone up to performance plus brand, and present them in tabular form:

In [ ]:
cpu_reg_short1 = smf.ols("log_price ~ log_performance", data=cpu_data).fit()
cpu_reg_short2 = smf.ols("log_price ~ C(brand)", data=cpu_data).fit()
cpu_reg_long1 = smf.ols("log_price ~ log_performance + C(brand)",
                        data=cpu_data).fit()

results_table = summary_col([cpu_reg_short1, cpu_reg_short2, cpu_reg_long1],
                            stars=True,
                            model_names=['(1)', '(2)', '(3)'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)


### A categorical variable on its own

Column (2) deserves a closer look, because it is something we have not seen yet: a regression whose only explanatory variable is categorical. With a single categorical regressor, the coefficients are just group means in disguise: the intercept is the mean of log price for the base category (AMD), and the Intel coefficient is the difference between the Intel mean and the AMD mean. Let's verify both claims against a simple groupby:

In [ ]:
print(cpu_data.groupby('brand')['log_price'].mean())

Notice that the estimate on `brand` decreases substantially from column (2) to column (3). Why is that? Consider the output below, which compares average (log) performance by brand:

In [ ]:
cpu_data.groupby('brand')['log_performance'].mean()

Finally, let's collect predictions from the long regression. The `predict()` method applies the fitted equation to each row, and the residual is the gap between the actual and fitted values, the part of each CPU's price that performance and brand together do not explain:

In [ ]:
cpu_data['fitted'] = cpu_reg_long1.predict(cpu_data)
cpu_data['resid'] = cpu_reg_long1.resid

print(cpu_data.head())

## Regression in finance: estimating beta

In finance, a stock's **beta** measures how it moves with the overall market: a beta of 1 means the stock tends to move one-for-one with the market, a beta of 2 means a 1% market move comes with a 2% move in the stock on average, and a beta near 0 means the stock largely ignores the market. Beta is a direct application of everything in this notebook, because beta *is* a regression slope: regress the stock's returns on the market's returns, and the coefficient on the market return is beta. 

Let's estimate three betas with real data, chosen to cover the spectrum: **Tesla** (TSLA), a volatile growth stock; **Walmart** (WMT), a large retailer; and **SPDR Gold Shares** (GLD), an exchange-traded fund that holds gold. We use the S&P 500 index as the market. (Using the ETF rather than gold itself is a deliberate choice: gold trades nearly around the clock, but GLD trades during NYSE hours, so its daily returns are measured over exactly the same window as the market's.)

In [ ]:
market_data = pd.read_pickle('market_data.pkl')

market_data.head()

Price levels are not comparable across assets (the S&P 500 sits near 6,000 while Walmart trades near 100), so we work with daily **returns**: each day's percent change from the day before. `pct_change()` computes this for every column at once, and we drop the first row (the 2024 anchor day), which has no prior day in the dataset to compare against:

In [ ]:
market_returns = (
    market_data
    .set_index('date')
    .pct_change()
    .dropna()
    .reset_index()
)

market_returns.head()

As always, we plot before we regress. Let's reshape the returns to long format with `melt()` (as in [Working with Data](../04_working_data)) so that `lmplot` can draw one panel per asset, each with its own OLS line. The slope of each red line *is* that asset's beta:

In [ ]:
returns_long = market_returns.melt(id_vars=['date', 'sp500'],
                                   var_name='asset',
                                   value_name='asset_return')
returns_long['asset'] = returns_long['asset'].str.upper()

grid = sns.lmplot(data=returns_long,
                  x='sp500',
                  y='asset_return',
                  col='asset',
                  ci=None,
                  scatter_kws={'alpha': 0.4},
                  line_kws={'color': 'red'},
                  height=4)

grid.set_axis_labels('S&P 500 Daily Return', 'Asset Daily Return')
grid.set_titles('{col_name}')
grid.figure.suptitle('Daily Returns vs. the Market (2025)', y=1.05)

plt.show()

Three very different pictures: the Tesla cloud is steep and stretched out, the Walmart cloud rises gently, and the gold cloud is essentially flat. Let's put numbers on the three slopes by estimating the regressions and collecting them in one table:

In [ ]:
tsla_beta_reg = smf.ols("tsla ~ sp500", data=market_returns).fit()
wmt_beta_reg = smf.ols("wmt ~ sp500", data=market_returns).fit()
gld_beta_reg = smf.ols("gld ~ sp500", data=market_returns).fit()

results_table = summary_col([tsla_beta_reg, wmt_beta_reg, gld_beta_reg],
                            stars=True,
                            model_names=['TSLA', 'WMT', 'GLD'],
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

Reading the table:

- **Tesla's beta is about 2.3.** On a day the market rises 1%, Tesla rises about 2.3% on average, a classic high-beta stock. The r-squared of about 0.48 says the market explains nearly half of Tesla's daily variation; the rest is the residual: everything that moves Tesla's price but is not captured by the market return, from delivery numbers and product news to plain noise.
- **Walmart's beta is about 0.6.** People buy groceries whether the economy booms or busts, so Walmart's business, and its stock, is less tied to the market's mood. This is what a **defensive** stock looks like. Note that low beta is not low quality: Walmart returned about 25% in 2025, beating the market's 16%.
- **Gold's beta is about 0.** This lack of co-movement, at least in this sample, is exactly why gold shows up in diversification arguments.
- **Beta is not performance.** The zero-beta asset was the runaway winner of 2025: gold rose about 64% while the S&P 500 rose about 16%. That return lived almost entirely in the intercept and the residuals; the fitted market model attributes essentially none of it to market exposure.

Two notes for the finance-literate. Practitioners usually estimate beta with *excess* returns (returns minus the risk-free rate); at a daily horizon the adjustment is tiny, so we skip it here. And beta is not a constant of nature; it changes with the sample period and the return frequency, which is one reason published betas for the same stock disagree.

## Binary outcomes and logistic regression

We are commonly tasked with predicting a **binary** outcome variable, one that takes only the values 0 or 1: fraud or not fraud, default or no default, win or loss, bankrupt or not bankrupt. A variable with only two possible values cannot be normally distributed, and what we are really estimating is the *probability* of the outcome, based on predictor variables. OLS methods will work in this setting (they are retitled 'linear probability models', or LPM), but a problem with OLS here is that some predictions can fall outside the range of 0 to 1. Special regression techniques, like **logistic regression**, are often better suited to this setting because they confine predicted probabilities to the range of 0 to 1.

### What could you use logistic regression for?

- Predicting whether a loan will be repaid.
- Predicting whether a client will have a restatement or a regulatory issue.
- Predicting whether someone will file an insurance claim.
- What other applications can you think of?

Let's illustrate with some simulated data, where each observation has a continuous predictor and a binary outcome:

In [ ]:
logit_data = pd.read_csv('logit_data.csv')

logit_data.head()

Next, let's plot this dataset and fit an OLS (linear probability) regression line, drawn in blue here to set it apart from the red logistic curve we draw next. Notice that nothing stops this line from predicting probabilities below 0 or above 1:

In [ ]:
sns.lmplot(data=logit_data,
           x='predictor',
           y='important_outcome',
           scatter_kws={'alpha': 0.5},
           ci=None,
           line_kws={'color': 'blue'},
           height=6,
           aspect=1.2
           )

plt.title("Prediction of Important Outcome")
plt.xlabel("Predictor")
plt.ylabel("Important Outcome")

plt.show()

Now let's draw the same data with a logistic regression line instead. The S-shaped curve flattens out near 0 and 1, so the predicted probabilities stay in range:

In [ ]:
sns.lmplot(data=logit_data,
           x='predictor',
           y='important_outcome',
           scatter_kws={'alpha': 0.5},
           ci=None,
           logistic=True,
           line_kws={'color': 'red'},
           height=6,
           aspect=1.2
           )

plt.title("Prediction of Important Outcome")
plt.xlabel("Predictor")
plt.ylabel("Important Outcome")

plt.show()

Finally, here is an illustration of how to estimate a logistic regression and how to store the predictions. The syntax mirrors OLS regression: we simply swap `ols()` for `logit()`:

In [ ]:
outcome_logit = smf.logit("important_outcome ~ predictor", data=logit_data).fit()

print(outcome_logit.summary())

Next, let's add a new column with the predicted probabilities. Each prediction is the model's estimated probability that the outcome occurs for that observation:

In [ ]:
logit_data['predicted'] = outcome_logit.predict(logit_data)

logit_data.head()

Let's try to do this with the CPU dataset by predicting whether a processor is Intel based on its performance. Given the brand differences we saw above, how well do you expect performance to separate the two brands?

In [ ]:
cpu_data['is_intel'] = (cpu_data['brand'] == 'Intel').astype(int)

sns.lmplot(data=cpu_data,
           x='log_performance',
           y='is_intel',
           scatter_kws={'alpha': 0.2},
           ci=None,
           height=6,
           aspect=1.2,
           logistic=True,
           line_kws={'color': 'red'}
           )

plt.title("Probability CPU is Intel vs. Performance (Logistic Fit)")
plt.xlabel("Log Performance")
plt.ylabel("P(Intel)")

plt.show()

The curve climbs with performance, but it does so gradually: faster chips are more likely to be Intel, yet performance alone leaves plenty of overlap between the brands.


## Wrapping up

Look back at what you did in this notebook: you measured correlations, fit simple and multiple regressions with `ols()`, read coefficient tables and r-squared values, used Anscombe's quartet to see why plotting comes before regressing, handled non-linear relations with squared terms and logs, folded categorical variables into a model, estimated market betas from real stock returns, and estimated logistic regressions for binary outcomes. These ideas return in the next module, [Causal and Statistical Inference](../07_causal_statistical_inference), where we ask the questions this notebook set aside: how confident can we be in an estimated coefficient, and when can a regression support the claim that x actually *causes* y?